# Amortized Analysis

Sometimes an operation is *usually* cheap but *occasionally* expensive. If we just use worst-case analysis, we overestimate badly.

**Amortized analysis** finds the average cost per operation over a worst-case sequence -- not the average case, but the worst-case total divided by the number of operations.

| Analysis type | What it measures |
|---------------|------------------|
| Worst-case | Cost of the single most expensive operation |
| Average-case | Expected cost assuming random inputs |
| **Amortized** | Average cost per operation in the worst-case sequence |

**Key insight:** Expensive operations can only happen *because* many cheap operations happened first. The cheap ones "pay" for the expensive ones.

## Three Methods

1. **Aggregate method** -- compute total cost T(n), divide by n
2. **Accounting method** -- assign a "charge" to each operation, bank the surplus
3. **Potential method** -- define a potential function on the data structure (more advanced, not covered here)

We'll use the **dynamic array** (Python's `list.append()`) as our running example.

# The Problem: Dynamic Array Append

A dynamic array starts with capacity 1. When full, it allocates a new array of **double** the size and copies everything over.

| Append # | Array state | Cost |
|----------|-------------|------|
| 1 | `[1]` capacity 1 -> full, copy to capacity 2 | 1 + 1 = 2 |
| 2 | `[1,2]` capacity 2 -> full, copy to capacity 4 | 1 + 2 = 3 |
| 3 | `[1,2,3]` capacity 4 | 1 |
| 4 | `[1,2,3,4]` capacity 4 -> full, copy to capacity 8 | 1 + 4 = 5 |
| 5-8 | fill capacity 8 | 1 each |
| 9 | capacity 8 -> full, copy to capacity 16 | 1 + 8 = 9 |

**Worst-case per operation:** O(n) -- when we have to copy everything.

**Naive total for n appends:** n x O(n) = O(n^2) -- way too pessimistic!

The copy only happens at powers of 2. Most appends cost just 1.

# Method 1: Aggregate

Compute the total cost of n appends, then divide by n.

### Step 1: Define the cost of each operation

Let t(i) = cost of the ith append:

- **Case 1:** No reallocation needed. Just assign the element. t(i) = 1
- **Case 2:** Array is full before the insert (i = 2^k for some k >= 0, i.e., size equals capacity). Must allocate new array, copy 2^k elements, then assign. t(i) = 2^k + 1

### Step 2: Compute total cost T(n)

```
T(n) = sum of t(i) for i = 1 to n
```

Every operation pays at least 1 (the assignment), so we can split:

```
T(n) = n  +  (sum of copy costs)
       ^         ^
       |         |
  n assignments   copies only happen when i = 2^k + 1
```

### Step 3: Sum the copy costs

Copies happen at i = 1, 2, 4, 8, 16, ... (i.e., when i = 2^k). At each, we copy 2^k elements.

How many times can this happen? At most floor(log2(n)) + 1 times, since 2^k <= n means k <= log2(n).

So the copy costs form a geometric series:

```
Copy costs = 2^0 + 2^1 + 2^2 + ... + 2^floor(log2(n))
```

Using the geometric series formula: sum of 2^j for j = 0 to m = 2^(m+1) - 1

```
Copy costs = 2^(floor(log2(n)) + 1) - 1
           <= 2n - 1          (since 2^floor(log2(n)) <= n)
```

### Step 4: Combine

```
T(n) = n + copy costs
     <= n + (2n - 1)
     = 3n - 1
     <= 3n
```

### Step 5: Amortized cost

```
Amortized cost = T(n) / n <= 3n / n = 3 = O(1)
```

Each append costs **O(1) amortized**, even though individual appends can cost O(n).

In [ ]:
# Let's verify empirically
def simulate_appends(n):
    """Simulate n appends to a dynamic array, tracking total cost."""
    capacity = 1
    size = 0
    total_cost = 0
    for i in range(1, n + 1):
        cost = 1  # assignment
        if size == capacity:
            cost += capacity  # copy all elements
            capacity *= 2
        size += 1
        total_cost += cost
    return total_cost

for n in [100, 1000, 10000, 100000]:
    total = simulate_appends(n)
    print(f'n={n:>6}  total={total:>7}  amortized={total/n:.2f}')

# Method 2: Accounting (Banker's Method)

Assign each operation a **charge** (what we "pay" upfront). If the actual cost is less, bank the surplus. If more, withdraw from the bank.

**Rule:** The bank balance must never go negative.

### Why charge $3?

Between two expensive operations (at i = 2^(k-1) + 1 and i = 2^k + 1), there are 2^(k-1) - 1 cheap operations. Each cheap operation costs $1, so if we charge $3, we bank $2 per cheap operation.

Savings from cheap operations: 2 x (2^(k-1) - 1) = 2^k - 2

Cost of the next expensive operation: 2^k + 1

We pay $3 for the expensive operation itself, plus withdraw 2^k - 2 from the bank:

```
3 + (2^k - 2) = 2^k + 1  ✓  exactly enough!
```

### Step-by-step trace

| Append # | Actual cost | Charge | Bank change | Bank balance |
|----------|-------------|--------|-------------|-------------|
| 1 | 2 (assign + copy 1) | 3 | +1 | 1 |
| 2 | 3 (assign + copy 2) | 3 | 0 | 1 |
| 3 | 1 | 3 | +2 | 3 |
| 4 | 5 (assign + copy 4) | 3 | -2 | 1 |
| 5 | 1 | 3 | +2 | 3 |
| 6 | 1 | 3 | +2 | 5 |
| 7 | 1 | 3 | +2 | 7 |
| 8 | 9 (assign + copy 8) | 3 | -6 | 1 |

Bank never goes negative. After each expensive operation, the balance resets to 1, and cheap operations build it back up before the next expensive one.

**Amortized cost = $3 = O(1)**

The accounting method is more flexible than aggregate -- it can assign different charges to different operation types (useful when analyzing data structures with multiple operations like push/pop on a stack).

In [ ]:
# Verify the bank never goes negative
def verify_accounting(n, charge=3):
    """Verify that charging $charge per append keeps bank >= 0."""
    capacity = 1
    size = 0
    bank = 0
    for i in range(1, n + 1):
        cost = 1
        if size == capacity:
            cost += capacity
            capacity *= 2
        size += 1
        bank += charge - cost
        assert bank >= 0, f'Bank negative at append {i}: {bank}'
    print(f'n={n}, charge=${charge}: bank never negative (final balance: ${bank})')

verify_accounting(1000)
verify_accounting(10000)

# Method 3: Potential (Physics Method)

<details>
<summary><strong>Click to expand -- more advanced, uses a potential function like energy in physics</strong></summary>

### Intuition

Think of a swinging pendulum. At the top, velocity is zero but potential energy is high. At the bottom, velocity is maximum but potential energy is low. Energy converts between forms.

Similarly, define a **potential function** Phi on the data structure's state. When cheap operations run, potential builds up. When an expensive operation runs, potential drops and "pays" for it.

### Definition

For each operation i with real cost t(i), the amortized cost is:

```
a(i) = t(i) + Phi(after) - Phi(before)
```

If the operation is expensive, Phi drops (Phi(after) < Phi(before)), reducing the amortized cost.  
If the operation is cheap, Phi rises, "storing energy" for later.

The total amortized cost telescopes:

```
sum of a(i) = sum of t(i) + Phi(final) - Phi(initial)
```

If Phi(final) >= Phi(initial) (which we ensure by choosing Phi >= 0), then sum of a(i) >= sum of t(i), so the amortized cost is an upper bound on the real cost.

### Applied to Dynamic Array

Define: **Phi = 2 x length - capacity + 1**

This is always >= 0 because we double capacity when full, so after a resize, length = capacity/2 + 1, giving Phi = 2(capacity/2 + 1) - capacity + 1 = 3. Between resizes, length grows from capacity/2 + 1 to capacity, so Phi ranges from 3 to capacity + 1 -- always >= 0. At initialization (length=0, capacity=1), Phi = 0.

**Cheap operation** (no resize): length increases by 1, capacity unchanged.

```
a(i) = t(i) + Phi(after) - Phi(before)
     = 1 + (2(length+1) - capacity + 1) - (2*length - capacity + 1)
     = 1 + 2
     = 3
```

**Expensive operation** (resize at i = 2^k): length goes from 2^k to 2^k + 1, capacity goes from 2^k to 2^(k+1).

```
Phi(before) = 2(2^k) - 2^k + 1 = 2^k + 1
Phi(after)  = 2(2^k + 1) - 2^(k+1) + 1 = 3

a(i) = t(i) + Phi(after) - Phi(before)
     = (2^k + 1) + 3 - (2^k + 1)
     = 3
```

Both cases give amortized cost = **3 = O(1)**.

### Potential vs Accounting

They're closely related, but:

- **Accounting:** bank balance depends on the *history* of operations (how much was deposited/withdrawn)
- **Potential:** Phi depends only on the *current state* of the data structure, not how we got there

This makes the potential method more powerful for complex data structures where the same state can be reached via different operation sequences.

</details>

# Where Amortized Analysis Appears

| Data structure / Operation | Worst-case | Amortized | Why |
|---------------------------|-----------|-----------|-----|
| Dynamic array `append` | O(n) | O(1) | Doubling strategy |
| Hash table `insert` | O(n) | O(1) | Resizing when load factor exceeded |
| [Union-Find](../graphs/union-find.ipynb) `find`/`union` | O(log n) | O(α(n)) | Path compression flattens over time |
| Splay tree operations | O(n) | O(log n) | Frequent nodes move to root |
| Stack with multipop | O(n) | O(1) | Each element pushed/popped at most once |

## Common Misconception

Amortized != average-case.

- **Average-case** assumes a probability distribution over inputs. It can be wrong if the distribution is wrong.
- **Amortized** is a worst-case guarantee over any sequence. No assumptions about input distribution. It's always correct.

When someone says Python's `list.append` is O(1), they mean O(1) *amortized* -- guaranteed over any sequence of appends.